<a href="https://colab.research.google.com/github/mbahramii/conveyor-stone-detector/blob/main/Stone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
import numpy as np
from pathlib import Path

# Input and output video paths
INPUT_VIDEO  = "/content/input.mp4"
OUTPUT_VIDEO = "/content/output.mp4"

# Open input video file
cap = cv2.VideoCapture(INPUT_VIDEO)

# Extract video properties (fps, width, height, total frames)
fps    = int(cap.get(cv2.CAP_PROP_FPS))
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Define video writer for saving processed output
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

print(f"✅ Video: {width}x{height} | {fps}fps | {total} frame")

frame_count = 0

# Process video frame by frame
while cap.isOpened():
    ret, img = cap.read()

    # Stop if video ends or frame is not read correctly
    if not ret:
        break

    # Get frame dimensions
    h, w = img.shape[:2]

    # Define region of interest (ROI) zone where stones are expected
    zone = np.array([
        [int(w * 0.62), int(h * 0.64)],
        [int(w * 0.98), int(h * 0.64)],
        [int(w * 0.98), int(h * 0.98)],
        [int(w * 0.62), int(h * 0.98)],
    ], dtype=np.int32)

    # Create mask for ROI to ignore irrelevant background
    roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(roi_mask, [zone], 255)

    # Convert frame to grayscale for easier processing
    gray     = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Apply CLAHE to improve contrast in uneven lighting conditions
    clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    # Apply Gaussian blur to reduce noise
    blurred  = cv2.GaussianBlur(enhanced, (5, 5), 0)

    # Apply adaptive thresholding for foreground/background separation
    binary = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 51, 10)

    # Apply ROI mask to focus only on region of interest
    binary = cv2.bitwise_and(binary, binary, mask=roi_mask)

    # Morphological kernels for noise removal and shape refinement
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    k_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

    # Perform morphological closing and opening to clean segmentation
    binary  = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k_close, iterations=3)
    binary  = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  k_open,  iterations=2)

    # Invert binary image for contour detection
    binary_inv = cv2.bitwise_not(binary)

    # Apply ROI mask again after inversion
    binary_inv = cv2.bitwise_and(binary_inv, binary_inv, mask=roi_mask)

    # Detect contours (potential stones)
    cnts, _ = cv2.findContours(binary_inv, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)

    # Create visualization frame
    vis = img.copy()

    # Draw ROI zone on output frame
    cv2.polylines(vis, [zone], True, (0, 255, 255), 2)

    stone_found = False

    # If any contours exist, process the largest one
    if cnts:
        best = max(cnts, key=cv2.contourArea)

        # Filter small objects (noise removal)
        if cv2.contourArea(best) > 1000:

            # Compute rotated bounding box for detected stone
            rect = cv2.minAreaRect(best)
            box  = np.intp(cv2.boxPoints(rect))

            # Draw detected stone boundary
            cv2.drawContours(vis, [box], 0, (255, 0, 0), 2)

            # Label detected stone
            cv2.putText(vis, "Stone", (box[0][0], box[0][1]-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

            stone_found = True

    # Display detection status on frame
    status = "STONE ✓" if stone_found else "NO STONE"
    color  = (0, 255, 0) if stone_found else (0, 0, 255)

    cv2.putText(vis, status, (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)

    # Write processed frame to output video
    out.write(vis)

    frame_count += 1

    # Progress logging every 30 frames
    if frame_count % 30 == 0:
        print(f"  ⏳ {frame_count}/{total} Frame processed...")

# Release resources
cap.release()
out.release()

print(f"\n✅ Video saved successfully: {OUTPUT_VIDEO}")

In [ ]:
import cv2
import numpy as np

# Input and output video paths
INPUT_VIDEO  = "/content/input.mp4"
OUTPUT_VIDEO = "/content/output_area.mp4"

# Load video capture object
cap = cv2.VideoCapture(INPUT_VIDEO)

# Extract video properties (FPS, width, height, total frame count)
fps    = int(cap.get(cv2.CAP_PROP_FPS))
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Define video writer for saving processed output video
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

print(f"✅ Video: {width}x{height} | {fps}fps | {total} frame")

frame_count = 0

# Process video frame by frame
while cap.isOpened():
    ret, img = cap.read()

    # Stop loop if video ends or frame read fails
    if not ret:
        break

    # Get frame dimensions
    h, w = img.shape[:2]

    # Define region of interest (ROI) where stones are expected
    zone = np.array([
        [int(w * 0.62), int(h * 0.64)],
        [int(w * 0.98), int(h * 0.64)],
        [int(w * 0.98), int(h * 0.98)],
        [int(w * 0.62), int(h * 0.98)],
    ], dtype=np.int32)

    # Create empty mask for ROI extraction
    roi_mask = np.zeros((h, w), dtype=np.uint8)

    # Fill ROI polygon into mask
    cv2.fillPoly(roi_mask, [zone], 255)

    # Convert frame to grayscale for processing
    gray     = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Apply CLAHE to improve local contrast (handles uneven lighting)
    clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    # Apply Gaussian blur to reduce noise
    blurred  = cv2.GaussianBlur(enhanced, (5, 5), 0)

    # Apply adaptive thresholding for segmentation
    binary = cv2.adaptiveThreshold(
        blurred, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        51, 10
    )

    # Apply ROI mask to keep only relevant area
    binary = cv2.bitwise_and(binary, binary, mask=roi_mask)

    # Define morphological kernels for noise removal
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    k_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

    # Perform morphological closing (fill gaps) and opening (remove noise)
    binary  = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k_close, iterations=3)
    binary  = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  k_open,  iterations=2)

    # Invert binary image for contour detection
    binary_inv = cv2.bitwise_not(binary)

    # Apply ROI mask again after inversion
    binary_inv = cv2.bitwise_and(binary_inv, binary_inv, mask=roi_mask)

    # Find contours (potential stone regions)
    cnts, _ = cv2.findContours(binary_inv, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)

    # Create visualization copy of frame
    vis = img.copy()

    # Draw ROI zone on output frame
    cv2.polylines(vis, [zone], True, (0, 255, 255), 2)

    stone_found = False

    # If any contours are detected
    if cnts:

        # Select largest contour (most likely stone)
        best = max(cnts, key=cv2.contourArea)

        # Compute contour area in pixels
        area_px = cv2.contourArea(best)

        # Filter out small noise objects
        if area_px > 1000:

            # Get minimum area rotated bounding box
            rect = cv2.minAreaRect(best)

            # Convert box points to integer coordinates
            box  = np.intp(cv2.boxPoints(rect))

            # Extract width and height of detected object
            rw, rh = int(rect[1][0]), int(rect[1][1])

            # Draw bounding box around detected stone
            cv2.drawContours(vis, [box], 0, (255, 0, 0), 2)

            # Display area of detected object
            cv2.putText(vis, f"Area: {int(area_px)} px2",
                        (box[0][0], box[0][1] - 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

            # Display approximate size of object
            cv2.putText(vis, f"Size: {rw}x{rh} px",
                        (box[0][0], box[0][1] - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

            stone_found = True

    # Set status label based on detection result
    status = "STONE" if stone_found else "NO STONE"
    color  = (0, 255, 0) if stone_found else (0, 0, 255)

    # Draw status text on frame
    cv2.putText(vis, status, (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)

    # Write processed frame to output video
    out.write(vis)

    frame_count += 1

    # Print progress every 30 frames
    if frame_count % 30 == 0:
        print(f"  ⏳ {frame_count}/{total} فریم...")

# Release resources
cap.release()
out.release()

print(f"\n✅ Video saved successfully: {OUTPUT_VIDEO}")